In [1]:
from platform import python_version
print(python_version())

3.11.14


### Cluster with Tahoe or sc-GTP

### Huggingface: tahoebio/Tahoe-x1-embeddings

https://github.com/tahoebio/tahoe-x1

Tahoe-x1: Scaling Perturbation-Trained Single-Cell Foundation Models to 3 Billion Parameters


#### Memory

That's not a general "64 GB isn't enough" — swap is fully exhausted at 2.0G, which means something asked for tens of GB in one allocation. Given where you are in the pipeline, the culprit is almost certainly load_tahoe_de, and the arithmetic says so:

The DE table is ~4.09e9 rows over ~75k conditions × ~54k genes. 

Filtering to pancreas doesn't help much — roughly 
- 6 lines × 379 drugs × ~4 doses × 54k genes ≈ 5e8 rows, 
- materialised in pandas with gene/drug/cell_line_id as object-dtype strings (~200 B/row) before pivot_table ever runs. 
- That's >100 GB. full_Z and consensus_cluster are megabytes by comparison.



In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [8]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



(PosixPath('/home/flavio/uv/perturb_agent/data/single_cell'), True)

### Running prism

In [9]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

Loaded /home/flavio/uv/perturb_agent/data/single_cell/deconv.h5ad (6.8 MB)
1604


In [10]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

In [11]:
res.theta_type.columns

Index(['Acinar cell', 'B cell', 'Ductal cell type 1', 'Endocrine cell', 'Endothelial cell',
       'Fibroblast cell', 'Macrophage cell', 'Stellate cell', 'T cell', 'malignant'],
      dtype='object')

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [12]:
res.cell_type_expression("Ductal cell type 1").shape

(1604, 153)

In [13]:
res.cell_type_expression("Ductal cell type 2").shape

(1604, 153)

### 2. theta is now fixed -> expand Z to every gene

In [14]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

verbose=True
force=False

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        force=force, verbose=verbose)

force=False
verbose=True
fname = "count-matrix.txt"
fname_ad = fname.replace('.txt', '.h5ad')

adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

filename_ad = prism.root_singc / fname_ad
compression = "gzip"
# adata.write_h5ad(filename_ad, compression=compression)
print(f"AData saved as {filename_ad},  ({filename_ad.stat().st_size/1e6:.0f} MB), compressed with {compression}")


verbose=True
fname_celltype = "all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

ref, s2t = prism.pseudobulk_reference(adata)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file
Table opened ((27169, 153)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_matrix.tsv'
Table opened ((153, 4)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_metadata.tsv'
57,530 cells x 24,005 genes | obs: []
AData saved as /home/flavio/uv/perturb_agent/data/single_cell/count-matrix.h5ad,  (338 MB), compressed with gzip
all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 24

In [15]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref)

In [16]:
dic = {}

for cell_state in res.states:
    print(cell_state)
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z

Fibroblast cell
Stellate cell
Macrophage cell
Endothelial cell
T cell
B cell
Ductal cell type 2
Endocrine cell
Ductal cell type 1
Acinar cell


In [17]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key]

Fibroblast cell


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
A1BG,6.551e-01,4.418e-01,6.680e-01,2.693e-01,4.914e-01,4.378e-01,1.043e+00,8.103e-01,9.495e-01,7.379e-01,...,NaN,0.772,0.328,NaN,NaN,NaN,1.515e-01,NaN,2.648e-01,NaN
A1BG-AS1,3.150e+00,1.251e+00,1.249e+00,2.249e+00,1.408e+00,1.848e+00,3.725e+00,1.988e+00,2.017e+00,2.263e+00,...,NaN,6.944,9.022,NaN,NaN,NaN,7.544e+00,NaN,1.126e+00,NaN
A1CF,3.463e+00,1.384e+00,2.188e+00,1.346e-01,1.305e+00,8.334e-02,1.176e+00,2.169e+00,1.218e+00,3.142e+00,...,NaN,1.033,3.148,NaN,NaN,NaN,1.032e+01,NaN,1.701e+00,NaN
A2M,1.344e+03,1.258e+03,1.165e+03,8.277e+02,1.477e+03,8.272e+02,1.130e+03,1.373e+03,2.035e+03,2.286e+03,...,NaN,1083.582,1060.576,NaN,NaN,NaN,2.068e+03,NaN,2.003e+03,NaN
A2M-AS1,6.254e+00,7.056e+00,2.835e+00,2.690e+00,5.443e+00,7.466e+00,6.568e+00,7.522e+00,6.423e+00,1.268e+01,...,NaN,2.626,7.625,NaN,NaN,NaN,1.319e+01,NaN,2.975e+00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,8.548e-09,2.341e-08,6.967e-08,7.866e-09,2.045e-08,2.942e-08,6.440e-08,2.001e-08,1.809e-07,1.226e-08,...,NaN,0.021,0.000,NaN,NaN,NaN,7.822e-08,NaN,6.635e-09,NaN
ZYG11B,1.060e+02,1.070e+02,1.151e+02,8.354e+01,1.278e+02,9.075e+01,1.159e+02,1.173e+02,9.685e+01,1.466e+02,...,NaN,53.810,83.431,NaN,NaN,NaN,1.003e+02,NaN,5.187e+01,NaN
ZYX,7.451e+01,5.875e+01,7.764e+01,1.232e+02,4.028e+01,7.744e+01,5.810e+01,6.771e+01,7.150e+01,5.684e+01,...,NaN,87.890,61.471,NaN,NaN,NaN,3.394e+01,NaN,3.417e+02,NaN
ZZEF1,1.460e+02,1.129e+02,9.541e+01,1.231e+02,1.044e+02,1.199e+02,1.055e+02,1.418e+02,1.251e+02,1.251e+02,...,NaN,129.316,99.746,NaN,NaN,NaN,5.434e+02,NaN,8.607e+01,NaN


### Ductal 2 - malignant

In [18]:
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")

In [19]:
for g in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"]:
    if g in gfull:
        print(g, prism.gene_compartment_share(Zfull, gfull, res, g).head(3).round(3).to_dict())

FAM83A-AS1 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
HOXA10-AS {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
HOXB-AS3 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
MIR7-3HG {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}


In [20]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]

prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [21]:
[g for g in prog1 if g in df_bulk.index]

['FAM83A-AS1', 'HOXA10-AS', 'HOXB-AS3', 'HOXB-AS4', 'MIR7-3HG']

### present in the scRNA reference?

In [22]:
  
[g for g in prog1 if g in ref.columns]

['FAM83A-AS1', 'HOXA10-AS', 'HOXB-AS3', 'MIR7-3HG']

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [23]:
set(ref.index.to_list())

{'Acinar cell',
 'B cell',
 'Ductal cell type 1',
 'Ductal cell type 2',
 'Endocrine cell',
 'Endothelial cell',
 'Fibroblast cell',
 'Macrophage cell',
 'Stellate cell',
 'T cell'}

In [24]:
set(s2t.index.to_list())

{'Acinar cell',
 'B cell',
 'Ductal cell type 1',
 'Ductal cell type 2',
 'Endocrine cell',
 'Endothelial cell',
 'Fibroblast cell',
 'Macrophage cell',
 'Stellate cell',
 'T cell'}

In [25]:
adata.obs

,cluster,cell_type,cell_state
cell,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell
...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1


In [26]:
import re, numpy as np, pandas as pd

adata.obs["sample"] = adata.obs_names.to_series().str.extract(r"^([TN]\d+)_")[0].values
adata.obs["tissue"] = np.where(adata.obs["sample"].str.startswith("T"), "tumor", "normal")

print(adata.obs.groupby("tissue")["sample"].nunique())      # expect tumor 24, normal 11
print(pd.crosstab(adata.obs["cell_state"], adata.obs["tissue"]))

tissue
normal    11
tumor     24
Name: sample, dtype: int64
tissue              normal  tumor
cell_state                       
Acinar cell           1423    512
B cell                  31   2416
Ductal cell type 1    7671   2646
Ductal cell type 2       0  11315
Endocrine cell         270    459
Endothelial cell      3983   5134
Fibroblast cell        940   5802
Macrophage cell        559   4802
Stellate cell          623   5284
T cell                  44   3616


### Count Malignant Cells - accordingo to transcriptomics

In [27]:
d2 = adata.obs["cell_state"].eq("Ductal cell type 2")
print(len(d2))
d2[:5]

57530


cell
T1_AAACCTGAGATGTCGG    False
T1_AAACGGGGTCATGCAT    False
T1_AAAGATGCATGTTGAC    False
T1_AAAGATGGTCGAGTTT    False
T1_AAAGATGGTCTCTCTG    False
Name: cell_state, dtype: bool

In [28]:
is_t = adata.obs["tissue"].eq("tumor")
print(np.sum(is_t))

41986


In [29]:
adata.obs["cell_state"] = np.where(d2 &  is_t, "Malignant ductal",
                          np.where(d2 & ~is_t, "Ductal cell type 2 normal",
                                   adata.obs["cell_state"]))
adata.obs

,cluster,cell_type,cell_state,sample,tissue
cell,,,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell,T1,tumor
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell,T1,tumor
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell,T1,tumor
...,...,...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell,N11,normal
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell,N11,normal
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1,N11,normal


In [30]:
from collections import Counter

Counter(adata.obs["cell_state"] )

Counter({'Malignant ductal': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [31]:
adata.obs["cell_type"]  = np.where(adata.obs["cell_state"].eq("Malignant ductal"),
                                   "malignant", adata.obs["cell_type"])

Counter(adata.obs["cell_type"] )

Counter({'malignant': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [32]:
ref2, s2t = prism.pseudobulk_reference(adata, state_key="cell_state", type_key="cell_type")
s2t.to_dict()

{'Fibroblast cell': 'Fibroblast cell',
 'Stellate cell': 'Stellate cell',
 'Macrophage cell': 'Macrophage cell',
 'Endothelial cell': 'Endothelial cell',
 'T cell': 'T cell',
 'B cell': 'B cell',
 'Malignant ductal': 'malignant',
 'Endocrine cell': 'Endocrine cell',
 'Ductal cell type 1': 'Ductal cell type 1',
 'Acinar cell': 'Acinar cell'}

### LFC

calc_celltype_lfc() — each compartment vs the mean of the others, paired across samples by default. Paired is the right default here because every sample contributes every cell type, so pairing removes cohort/purity variance. This doubles as deconvolution QC: if the ductal compartment doesn't recover KRT19/TFF1/CEACAM6 and fibroblast doesn't recover COL1A1/POSTN, θ or the Peng reference is off and step 2 is meaningless.

### Critics

- Why not, for each cell type, tumor samples x normal samples
- Only Ductal 2 Tumor has no normal samples - to confirm


In [33]:
res.__dict__.keys()

dict_keys(['theta', 'theta_stage1', 'theta_type', 'tumor_purity', 'genes', 'Z', 'states'])

In [34]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

### Prism programs

In [35]:
Z_full, genes_full = prism.full_Z(res, df_bulk, ref)

### MalignantCluster

In [36]:
# del(MalignantCluster)

In [37]:
from libs.prism_malig_lib import MalignantCluster

In [38]:
type(res)

libs.prism_lib.DeconvResult

In [39]:
cbio.root_mprog_disease

PosixPath('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD')

In [40]:
root_mprog_cluster = create_dir(cbio.root_mprog_disease, 'cluster')

cell_name = "Ductal cell type 2"
kmax = 8
no_decouple = True
is_tahoe = True
'''
mc = MalignantCluster(prism=prism, res=res, df_bulk=df_bulk, ref=ref, 
                      root_mprog_cluster=root_mprog_cluster, 
                      organ="Pancreas", cell_name=cell_name, cell_types=None)
'''

import importlib, libs.prism_malig_lib as pml
importlib.reload(pml)
print(pml.__version__)

0.31.1


In [41]:
mc = pml.MalignantCluster(prism, res, df_bulk, ref, root_mprog_cluster, organ="Pancreas")

X, diag = mc.prepare_malignant_matrix(decouple_purity=False, keep_genes=mc.program1_panel, drop_pattern=r"^N-")
print(X.shape)
X.head(3)

excluded 22/153 samples by keep_samples/drop_pattern
(117, 2000)


,A1CF,AACS,AADAC,AATK,ABAT,ABCA12,ABCA7,ABCB9,ABCC3,ABCC6,...,ZNF774,ZNF787,ZNF792,ZNF816,ZNF888,ZNRF1,ZNRF2,ZSCAN29,ZSWIM5,ZWINT
T-C3L-02890,5.982,6.258,5.175,5.251,5.758,4.525,6.916,5.286,8.488,3.813,...,3.765,5.004,5.415,5.472,6.948,4.979,5.810,6.161,3.973,5.391
T-C3L-03635,4.757,5.724,1.766,4.362,5.829,6.391,5.053,3.572,9.914,2.841,...,4.566,4.348,5.478,6.968,7.315,4.711,6.219,6.158,3.293,5.177
T-C3L-02701,5.355,6.172,2.614,5.341,5.771,4.948,6.295,3.261,9.049,3.479,...,4.707,4.696,4.896,5.616,6.857,4.948,5.456,6.081,4.002,5.462


In [42]:
lista = [x for x in X.index if x.startswith('T-')]
X.shape[0], len(lista) == X.shape[0]

(117, True)

In [43]:
diag.keys()

dict_keys(['samples_excluded_by_filter', 'samples_dropped', 'n_genes_expressed', 'n_genes_share_not_computable', 'n_genes_share_ok', 'forced_genes_status', 'n_genes_kept', 'n_hvg', 'pc_theta_pearson_raw', 'pc_theta_pearson', 'decouple_purity', 'pc_theta_note', 'sample_mean_expr', 'sample_total_Z', 'theta_mal', 'n_samples_used', 'theta_excluded', 'theta_kept'])

In [44]:
diag["samples_excluded_by_filter"]

['N-C3L-04072',
 'N-C3L-00589',
 'N-C3L-03123',
 'N-C3L-04080',
 'N-C3L-00640',
 'N-C3N-01719',
 'N-C3L-07033',
 'N-C3L-00819',
 'N-C3L-07032',
 'N-C3L-01689',
 'N-C3N-01899',
 'N-C3N-00517',
 'N-C3N-03069',
 'N-C3N-02765',
 'N-C3L-07037',
 'N-C3N-02589',
 'N-C3N-02996',
 'N-C3L-02606',
 'N-C3N-03173',
 'N-C3N-02696',
 'N-TCGA-H6-8124',
 'N-TCGA-H6-A45N']

In [45]:
diag["pc_theta_pearson"] 

[-0.5957714316264521,
 -0.20525359661586903,
 0.22715645269047213,
 -0.359188494912212,
 -0.157968365575133]

In [46]:
diag["pc_theta_pearson_raw"]   # PC-vs-theta on logx (pre-decoupling)

[-0.5958662232635222,
 -0.20488740533784713,
 0.22757764165627772,
 -0.3592635404774974,
 -0.15953480236326065]

In [47]:
diag["pc_theta_note"]          # warns the decoupled version is ~0 by construction

'decouple_purity=False, so pc_theta_pearson and pc_theta_pearson_raw are the same matrix and both are informative: a large |r| on an early PC means the clustering is tracking tumour purity.'

In [48]:
diag["sample_mean_expr"]       # Xc.mean(axis=1) per sample

T-C3L-02890       6.188
T-C3L-03635       6.036
T-C3L-02701       6.146
T-C3L-04072       5.692
T-C3L-00589       6.043
                  ...  
T-TCGA-2L-AAQM    4.473
T-TCGA-3A-A9IR    4.186
T-TCGA-3A-A9IV    4.605
T-TCGA-2J-AABT    6.059
T-TCGA-H6-A45N    6.240
Length: 117, dtype: float32

In [49]:
diag["sample_total_Z"]         # ms.Z.sum(axis=1) per sample

T-C3L-02890       1.000e+06
T-C3L-03635       1.000e+06
T-C3L-02701       1.000e+06
T-C3L-04072       1.000e+06
T-C3L-00589       1.000e+06
                    ...    
T-TCGA-2L-AAQM    1.000e+06
T-TCGA-3A-A9IR    1.000e+06
T-TCGA-3A-A9IV    1.000e+06
T-TCGA-2J-AABT    1.000e+06
T-TCGA-H6-A45N    1.000e+06
Length: 117, dtype: float32

In [50]:
diag["theta_excluded"]

count    15.000
mean      0.291
std       0.418
min       0.000
25%       0.002
50%       0.060
75%       0.562
max       0.985
Name: Ductal cell type 2, dtype: float64

In [51]:
diag["theta_kept"]

count    130.000
mean       0.347
std        0.254
min        0.000
25%        0.145
50%        0.312
75%        0.493
max        1.000
Name: Ductal cell type 2, dtype: float64

### Inspecting vars

In [52]:
import inspect
print(pml.__version__)
print("drop_pattern" in inspect.signature(mc.prepare_malignant_matrix).parameters)

0.31.1
True


In [53]:
info = mc.inspect_de_schema(genes=X.columns)
print(info.keys())
info["columns"]

dict_keys(['shard_file', 'file_mb', 'total_rows_in_shard', 'columns', 'dtypes', 'head', 'distinct_gene_name', 'n_distinct_gene_name', 'distinct_baseMean', 'n_distinct_baseMean', 'distinct_log2FoldChange', 'n_distinct_log2FoldChange', 'distinct_lfcSE', 'n_distinct_lfcSE', 'distinct_stat', 'n_distinct_stat', 'distinct_pvalue', 'n_distinct_pvalue', 'distinct_padj', 'n_distinct_padj', 'distinct_plate', 'n_distinct_plate', 'distinct_n_cells_trt', 'n_distinct_n_cells_trt', 'distinct_n_cells_ctrl', 'n_distinct_n_cells_ctrl', 'distinct_Cell_ID_Cellosaur', 'n_distinct_Cell_ID_Cellosaur', 'distinct_Cell_ID_DepMap', 'n_distinct_Cell_ID_DepMap', 'distinct_drug', 'n_distinct_drug', 'distinct_concentration', 'n_distinct_concentration', 'distinct_concentration_unit', 'n_distinct_concentration_unit', 'distinct_Cell_Name_Vevo', 'n_distinct_Cell_Name_Vevo', 'cell_line_metadata_columns', 'MATCH cell_line_metadata.Cell_ID_Cellosaur -> DE.Cell_ID_Cellosaur', 'MATCH cell_line_metadata.cell_name -> DE.Cell_N

['gene_name',
 'baseMean',
 'log2FoldChange',
 'lfcSE',
 'stat',
 'pvalue',
 'padj',
 'plate',
 'n_cells_trt',
 'n_cells_ctrl',
 'Cell_ID_Cellosaur',
 'Cell_ID_DepMap',
 'drug',
 'concentration',
 'concentration_unit',
 'Cell_Name_Vevo']

In [54]:
info["matches"]

['MATCH cell_line_metadata.Cell_ID_Cellosaur -> DE.Cell_ID_Cellosaur',
 'MATCH cell_line_metadata.cell_name -> DE.Cell_Name_Vevo',
 'MATCH query genes -> DE.gene_name']

In [55]:
info["dtypes"]

{'gene_name': 'object',
 'baseMean': 'float32',
 'log2FoldChange': 'float32',
 'lfcSE': 'float32',
 'stat': 'float32',
 'pvalue': 'float32',
 'padj': 'float32',
 'plate': 'object',
 'n_cells_trt': 'int64',
 'n_cells_ctrl': 'int64',
 'Cell_ID_Cellosaur': 'object',
 'Cell_ID_DepMap': 'object',
 'drug': 'object',
 'concentration': 'float32',
 'concentration_unit': 'object',
 'Cell_Name_Vevo': 'object'}

In [56]:
info["resolved_columns"]

{'gene': 'gene_name',
 'stat': 'stat',
 'cell_line': 'Cell_ID_Cellosaur',
 'drug': 'drug'}

In [57]:
info["numeric_profile"]      # min / max / mean / frac_negative / n_unique


,min,max,mean,frac_negative,n_unique
baseMean,0.000,95136.398,36.460,0.000,69136
log2FoldChange,-4.858,7.180,0.081,0.232,68458
lfcSE,0.007,4.425,1.248,0.000,68449
stat,-46.548,73.326,0.019,0.232,68488
pvalue,0.000,1.000,0.470,0.000,68433
padj,0.000,1.000,0.586,0.000,23253
n_cells_trt,1378.000,2165.000,1745.045,0.000,4
n_cells_ctrl,4862.000,4862.000,4862.000,0.000,1
concentration,0.050,0.050,0.050,0.000,1


In [58]:
info["signed_candidates"]

['log2FoldChange', 'stat']

In [59]:
cov = mc.index_coverage()
cov["n_lines_seen"], cov["n_lines_in_metadata"]

(50, 102)

In [60]:
cov["block_probe_counts"]        # min probes per block

count    50.0
mean      3.4
std       0.5
min       3.0
25%       3.0
50%       3.0
75%       4.0
max       4.0
Name: count, dtype: float64

In [61]:
lista = cov["in_metadata_not_in_index"]
len(lista), lista[:5]

(52, ['CVCL_0025', 'CVCL_0031', 'CVCL_0039', 'CVCL_0060', 'CVCL_0078'])

In [62]:
cov["n_lines_seen"], cov["n_lines_in_metadata"]

(50, 102)

### state feasibility

In [63]:
X.iloc[:5, :10]

,A1CF,AACS,AADAC,AATK,ABAT,ABCA12,ABCA7,ABCB9,ABCC3,ABCC6
T-C3L-02890,5.982,6.258,5.175,5.251,5.758,4.525,6.916,5.286,8.488,3.813
T-C3L-03635,4.757,5.724,1.766,4.362,5.829,6.391,5.053,3.572,9.914,2.841
T-C3L-02701,5.355,6.172,2.614,5.341,5.771,4.948,6.295,3.261,9.049,3.479
T-C3L-04072,1.835,6.498,1.210,4.544,5.335,6.568,5.315,4.891,9.434,2.774
T-C3L-00589,4.656,6.309,5.162,5.493,5.485,6.388,6.047,4.658,9.567,4.075


In [64]:
dfs = mc.state_feasibility(n_samples=len(X), n_axes=3, levels=3)
dfs

,n_axes,levels,cells,mean_per_cell,usable
0,1,2,2,58.5,True
1,1,3,3,39.0,True
2,2,2,4,29.2,True
3,2,3,9,13.0,False
4,3,2,8,14.6,False
5,3,3,27,4.3,False


### cmap

Two things needed: 

- marker sets, and a check on which compartments are even scoreable 
- θ for B cells and endothelium in PDAC bulk is often 1–3%, where Z is mostly prior.

In [65]:
mc.df_theta.columns

Index(['Fibroblast cell', 'Stellate cell', 'Macrophage cell', 'Endothelial cell', 'T cell',
       'B cell', 'Ductal cell type 2', 'Endocrine cell', 'Ductal cell type 1', 'Acinar cell'],
      dtype='object')

In [66]:
cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
    "immune":     "T cell",
    "humoral":    "B cell",
    "ductal":     "Ductal cell type 1",
    "acinar":     "Acinar cell",
}

rd = mc.compartment_readiness(cmap)
rd[["compartment","theta_median","n_genes_after_share","marker_coverage","verdict"]]

,compartment,theta_median,n_genes_after_share,marker_coverage,verdict
0,malignant,2.918e-01,9583.0,"{'basal': 12, 'classical': 12, 'emt': 1, 'prolif': 6}",CAUTION - thin programs: ['emt']
1,fibroblast,3.638e-01,10757.0,"{'myCAF': 8, 'iCAF': 9, 'apCAF': 0}",CAUTION - thin programs: ['apCAF']
2,macrophage,1.680e-02,384.0,"{'M1': 1, 'TAM': 7, 'SPP1': 2}",DROP - theta median 0.017 too low
3,endothelial,2.330e-02,314.0,"{'tip_angio': 2, 'lymphatic': 1, 'activated': 1}",DROP - theta median 0.023 too low
4,immune,0.000e+00,NaN,NaN,matrix failed: T cell: only 0 samples x 0 genes survive
5,humoral,7.000e-04,NaN,NaN,matrix failed: B cell: only 0 samples x 0 genes survive
6,ductal,6.800e-03,166.0,"{'normal_duct': 2, 'ADM': 1}",DROP - theta median 0.007 too low
7,acinar,0.000e+00,1326.0,"{'acinar_identity': 7, 'stress': 3}",DROP - theta median 0.000 too low


In [67]:
cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
}

In [68]:
scores, cov = mc.program_scores(compartment_map=cmap, samples=X.index)   # tumours only

In [69]:
mat = [x for x in scores.index if x.startswith('T-')]
len(mat) == len(X), len(mat)

(True, 117)

In [70]:
scores

,malignant.basal,malignant.classical,malignant.emt,malignant.prolif,fibroblast.myCAF,fibroblast.iCAF,fibroblast.apCAF,macrophage.M1,macrophage.TAM,macrophage.SPP1,endothelial.tip_angio,endothelial.lymphatic,endothelial.activated,malignant.axis_basal_minus_classical,fibroblast.axis_myCAF_minus_iCAF
T-C3L-00277,-0.054,0.265,0.224,0.055,-0.244,-0.449,-1.366,NaN,NaN,NaN,0.947,-0.479,-1.527,-0.319,0.205
T-C3L-00589,-0.198,0.014,0.173,-0.585,-0.306,-0.062,-0.848,-0.883,0.103,-0.497,-0.798,-0.485,-1.500,-0.212,-0.244
T-C3L-00625,-0.012,0.422,-0.108,0.033,-0.168,-0.144,-0.704,NaN,NaN,NaN,1.001,0.298,-0.635,-0.434,-0.024
T-C3L-00640,-0.666,0.037,-0.449,-0.659,-0.187,0.031,-0.203,0.083,-0.292,0.644,NaN,NaN,NaN,-0.703,-0.218
T-C3L-00819,0.443,0.149,-0.332,0.250,-0.019,-0.396,-0.454,NaN,NaN,NaN,NaN,NaN,NaN,0.294,0.378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T-TCGA-US-A776,-0.211,1.004,-0.709,1.454,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.215,NaN
T-TCGA-US-A77E,0.153,0.588,-0.589,-0.174,0.255,-0.525,-0.244,NaN,NaN,NaN,NaN,NaN,NaN,-0.435,0.780
T-TCGA-US-A77J,0.346,-0.440,0.129,-0.818,-0.625,1.385,1.006,0.135,-0.046,-0.275,-0.791,0.594,-0.034,0.786,-2.010
T-TCGA-XN-A8T5,-0.586,-0.417,-0.489,-1.320,-0.788,1.227,0.925,NaN,NaN,NaN,0.274,-0.213,0.211,-0.169,-2.016


In [71]:
cov

,compartment,cell_type,program,n_found,n_total,missing,r_with_theta
0,malignant,Ductal cell type 2,basal,12,12,[],0.0
1,malignant,Ductal cell type 2,classical,12,12,[],0.0
2,malignant,Ductal cell type 2,emt,6,6,[],0.0
3,malignant,Ductal cell type 2,prolif,6,6,[],0.0
4,fibroblast,Fibroblast cell,myCAF,8,8,[],0.0
5,fibroblast,Fibroblast cell,iCAF,9,9,[],0.0
6,fibroblast,Fibroblast cell,apCAF,4,5,[SAA3],0.0
7,macrophage,Macrophage cell,M1,5,7,"[NOS2, IL6]",-0.0
8,macrophage,Macrophage cell,TAM,7,7,[],0.0
9,macrophage,Macrophage cell,SPP1,5,5,[],0.0


In [72]:
R = mc.couple_compartments(scores)
print(R.shape)
R

(83, 6)


,program_a,program_b,r,p_perm,n,fdr
0,fibroblast.iCAF,endothelial.lymphatic,0.432,4.998e-04,76,0.010
1,malignant.prolif,fibroblast.iCAF,-0.474,4.998e-04,112,0.010
2,macrophage.SPP1,endothelial.tip_angio,0.488,4.998e-04,49,0.010
3,malignant.prolif,fibroblast.axis_myCAF_minus_iCAF,0.379,4.998e-04,112,0.010
4,malignant.classical,fibroblast.apCAF,0.264,5.497e-03,112,0.090
...,...,...,...,...,...,...
78,macrophage.M1,malignant.axis_basal_minus_classical,0.024,8.546e-01,59,0.898
79,endothelial.lymphatic,malignant.axis_basal_minus_classical,-0.016,8.886e-01,77,0.917
80,malignant.classical,endothelial.lymphatic,-0.015,8.951e-01,77,0.917
81,malignant.emt,macrophage.TAM,0.010,9.360e-01,59,0.947


In [73]:
disc = mc.discretize_axes(scores)
disc

,basal_minus_classical,myCAF_minus_iCAF
T-C3L-00277,low,high
T-C3L-00589,low,low
T-C3L-00625,low,low
T-C3L-00640,low,low
T-C3L-00819,high,high
...,...,...
T-TCGA-US-A776,low,NaN
T-TCGA-US-A77E,low,high
T-TCGA-US-A77J,high,low
T-TCGA-XN-A8T5,low,low


In [74]:
dic_cross  = mc.axis_crosstab(disc)
dic_cross.keys()

dict_keys(['joint_label', 'counts', 'n_cells_possible', 'n_cells_occupied', 'n_cells_usable', 'sparse_cells', 'crosstab', 'chi2_p', 'cramers_v', 'independence_note'])

In [75]:
dic_cross['joint_label']

T-C3L-00277        basal_minus_classical:low | myCAF_minus_iCAF:high
T-C3L-00589         basal_minus_classical:low | myCAF_minus_iCAF:low
T-C3L-00625         basal_minus_classical:low | myCAF_minus_iCAF:low
T-C3L-00640         basal_minus_classical:low | myCAF_minus_iCAF:low
T-C3L-00819       basal_minus_classical:high | myCAF_minus_iCAF:high
                                         ...                        
T-TCGA-US-A776      basal_minus_classical:low | myCAF_minus_iCAF:nan
T-TCGA-US-A77E     basal_minus_classical:low | myCAF_minus_iCAF:high
T-TCGA-US-A77J     basal_minus_classical:high | myCAF_minus_iCAF:low
T-TCGA-XN-A8T5      basal_minus_classical:low | myCAF_minus_iCAF:low
T-TCGA-YH-A8SY    basal_minus_classical:high | myCAF_minus_iCAF:high
Length: 117, dtype: object

In [76]:
dic_cross['counts']

basal_minus_classical:low | myCAF_minus_iCAF:low      29
basal_minus_classical:low | myCAF_minus_iCAF:high     28
basal_minus_classical:high | myCAF_minus_iCAF:high    28
basal_minus_classical:high | myCAF_minus_iCAF:low     27
basal_minus_classical:high | myCAF_minus_iCAF:nan      3
basal_minus_classical:low | myCAF_minus_iCAF:nan       2
Name: count, dtype: int64

In [77]:
T = mc.state_signatures(X, mc.axis_crosstab(disc)["joint_label"], sort_by="fdr_nominal")
T

excluded 29 program marker genes; 1971 remain


,cluster,gene,lfc,stat,p_nominal,fdr_nominal,sd_cluster,sd_rest,welch_df,n_cluster,direction,rank,state
0,0,CDK1,7.403e-01,2.811e+00,4.632e-06,0.004,0.625,0.795,58.440,28,up,1,basal_minus_classical:high | myCAF_minus_iCAF:high
1,0,COLGALT2,-1.393e+00,-3.507e+00,7.121e-06,0.004,1.224,1.455,54.474,28,down,2,basal_minus_classical:high | myCAF_minus_iCAF:high
2,0,LEMD1,1.501e+00,3.638e+00,4.879e-06,0.004,1.292,1.535,54.489,28,up,3,basal_minus_classical:high | myCAF_minus_iCAF:high
3,0,MAD2L1,5.861e-01,2.525e+00,6.217e-06,0.004,0.524,0.542,47.735,28,up,4,basal_minus_classical:high | myCAF_minus_iCAF:high
4,0,CDCA5,7.817e-01,2.857e+00,1.003e-05,0.004,0.728,0.692,44.385,28,up,5,basal_minus_classical:high | myCAF_minus_iCAF:high
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7879,3,GABRE,-1.433e-03,-3.783e-03,9.959e-01,0.998,1.246,1.360,53.000,29,down,1967,basal_minus_classical:low | myCAF_minus_iCAF:low
7880,3,C1orf116,-1.268e-04,-5.509e-04,9.992e-01,1.000,0.497,0.790,78.362,29,down,1968,basal_minus_classical:low | myCAF_minus_iCAF:low
7881,3,CXCL1,2.265e-04,6.665e-04,9.992e-01,1.000,1.011,1.308,62.957,29,up,1969,basal_minus_classical:low | myCAF_minus_iCAF:low
7882,3,HS3ST1,1.559e-04,5.536e-04,9.993e-01,1.000,0.730,1.054,70.871,29,up,1970,basal_minus_classical:low | myCAF_minus_iCAF:low


In [78]:
# factorial decomposition — the better read for a 2x2
R = mc.factorial_state_de(X, disc)
R.head(3)

excluded 29 program marker genes; 1971 remain


,beta_A_basal_minus_classical,p_A_basal_minus_classical,fdr_A_basal_minus_classical,beta_B_myCAF_minus_iCAF,p_B_myCAF_minus_iCAF,fdr_B_myCAF_minus_iCAF,beta_interaction,p_interaction,fdr_interaction,n_samples,cell_counts
gene,,,,,,,,,,,
A1CF,0.107,0.764,0.928,-0.403,0.256,0.630,-0.827,0.103,0.97,112,"{('high', 'high'): 28, ('high', 'low'): 27, ('low', 'high'): 28, ('low', 'lo..."
LEMD1-AS1,-0.166,0.344,0.728,-0.239,0.169,0.575,0.368,0.138,0.97,112,"{('high', 'high'): 28, ('high', 'low'): 27, ('low', 'high'): 28, ('low', 'lo..."
LETM2,0.121,0.519,0.838,-0.446,0.018,0.411,0.445,0.095,0.97,112,"{('high', 'high'): 28, ('high', 'low'): 27, ('low', 'high'): 28, ('low', 'lo..."


### no tumour–stroma interaction detected

Uniform p-values — that's a clean null, not a broken model. With 1971 genes, the smallest p under a true null is ~1/1971 ≈ 5e-4, and BH turns that into q ≈ 0.999. Exactly what you see. So: no tumour–stroma interaction detected.

The discriminating question is whether the main effects are non-null:

In [79]:
R.fdr_interaction.describe()

count    1971.000
mean        0.979
std         0.008
min         0.970
25%         0.970
50%         0.980
75%         0.986
max         1.000
Name: fdr_interaction, dtype: float64

In [80]:
Rfdr = R[R.fdr_interaction < 0.1]
print(Rfdr.shape)
Rfdr

(0, 11)


,beta_A_basal_minus_classical,p_A_basal_minus_classical,fdr_A_basal_minus_classical,beta_B_myCAF_minus_iCAF,p_B_myCAF_minus_iCAF,fdr_B_myCAF_minus_iCAF,beta_interaction,p_interaction,fdr_interaction,n_samples,cell_counts
gene,,,,,,,,,,,


In [81]:
print((R.filter(like="fdr_A_") < 0.05).sum())
print((R.filter(like="fdr_B_") < 0.05).sum())

fdr_A_basal_minus_classical    30
dtype: int64
fdr_B_myCAF_minus_iCAF    0
dtype: int64


In [82]:
R.filter(like="fdr_").describe()

,fdr_A_basal_minus_classical,fdr_B_myCAF_minus_iCAF,fdr_interaction
count,1.971e+03,1971.000,1971.000
mean,6.668e-01,0.703,0.979
std,2.784e-01,0.185,0.008
min,4.434e-04,0.074,0.970
25%,4.991e-01,0.547,0.970
50%,7.566e-01,0.692,0.980
75%,8.937e-01,0.872,0.986
max,9.995e-01,1.000,1.000


That's a much more interesting result than it looks, because of which matrix you ran it on.

X is the malignant compartment. So the two main effects mean different things:

A (134 genes) — the tumour axis predicts 134 malignant genes beyond its own defining markers, which were excluded. That's real validation: the basal/classical axis isn't noise. But it's a within-compartment test, so partly expected.
B (0 genes) — the stroma axis fails to predict malignant expression. That's not "the stroma axis is noise." It's already a crosstalk test, and it's null.

The cell you haven't tested is the interesting one: does the tumour axis predict fibroblast expression? Run the same model on the other compartment.

In [83]:
coh = pd.Series(np.where(scores.index.str.contains("TCGA"), "TCGA", "CPTAC"), index=scores.index)
scores.groupby(coh)["fibroblast.axis_myCAF_minus_iCAF"].describe()

,count,mean,std,min,25%,50%,75%,max
CPTAC,44.0,0.007,0.538,-1.844,-0.307,-0.013,0.349,1.146
TCGA,68.0,-0.005,0.892,-2.575,-0.411,0.220,0.579,2.074


In [84]:
from scipy.stats import levene, bartlett

col = "fibroblast.axis_myCAF_minus_iCAF"
a = scores.loc[coh == "TCGA",  col].dropna()
b = scores.loc[coh == "CPTAC", col].dropna()
print(len(a), len(b), a.std(), b.std())

levene(a, b), bartlett(a, b)

68 44 0.89244958844855 0.5382763106081508


(LeveneResult(statistic=4.802951786359293, pvalue=0.030519163128213476),
 BartlettResult(statistic=11.910999784096916, pvalue=0.0005580344174452796))

In [85]:
for col in scores.columns:
    a = scores.loc[coh=="TCGA", col].dropna()
    b = scores.loc[coh=="CPTAC", col].dropna()
    print(f"{col:45s} sd {a.std():.3f}/{b.std():.3f}  levene p={levene(a,b).pvalue:.4f}")

malignant.basal                               sd 0.638/0.630  levene p=0.9712
malignant.classical                           sd 0.898/0.513  levene p=0.0421
malignant.emt                                 sd 0.474/0.289  levene p=0.0116
malignant.prolif                              sd 1.009/0.576  levene p=0.0082
fibroblast.myCAF                              sd 0.500/0.362  levene p=0.0402
fibroblast.iCAF                               sd 0.614/0.396  levene p=0.0130
fibroblast.apCAF                              sd 0.746/0.457  levene p=0.0351
macrophage.M1                                 sd 0.615/0.590  levene p=0.9501
macrophage.TAM                                sd 0.321/0.302  levene p=0.4790
macrophage.SPP1                               sd 0.581/0.584  levene p=0.7915
endothelial.tip_angio                         sd 0.593/0.605  levene p=0.6764
endothelial.lymphatic                         sd 0.512/0.355  levene p=0.0387
endothelial.activated                         sd 0.742/0.695  le

In [86]:
# and the tumour axis, for contrast
levene(scores.loc[coh=="TCGA","malignant.axis_basal_minus_classical"],
       scores.loc[coh=="CPTAC","malignant.axis_basal_minus_classical"])

LeveneResult(statistic=0.08904319511238377, pvalue=0.7659355719018054)

In [87]:
X_fib = mc.compartment_matrix(cmap["fibroblast"], min_share=0.3, min_counts=1)
R_fib = mc.factorial_state_de(X_fib, disc)
print((R_fib.filter(like="fdr_A_") < 0.05).sum())    # tumour -> stroma crosstalk
print((R_fib.filter(like="fdr_B_") < 0.05).sum())

excluded 56 program marker genes; 10701 remain
fdr_A_basal_minus_classical    0
dtype: int64
fdr_B_myCAF_minus_iCAF    399
dtype: int64


In [88]:
js = mc.joint_states(scores)
js["summary"]

,pac,cophenetic,silhouette,sizes,min_frac
2,0.646,0.963,0.803,"[20, 92]",0.179
3,0.526,0.962,0.806,"[8, 12, 92]",0.071
4,0.528,0.856,0.702,"[8, 12, 45, 47]",0.071
5,0.406,0.855,0.68,"[8, 12, 25, 31, 36]",0.071
6,0.377,0.836,0.633,"[8, 12, 15, 20, 27, 30]",0.071


In [89]:
best = mc.choose_k(js["consensus"], min_cluster_frac=0.10)
best

2

In [90]:
mc.state_profile(js)

,malignant.axis_basal_minus_classical,fibroblast.axis_myCAF_minus_iCAF,n,mean_abs_score
k2,,,,
1,-0.292,0.219,92,0.569
2,1.342,-1.006,20,1.460


In [91]:
mc.axis_modality(scores)

,axis,n,delta_bic,min_component_weight,separation_sd,skew,verdict
0,malignant.axis_basal_minus_classical,117,17.6,0.203,1.84,1.39,bimodal
1,fibroblast.axis_myCAF_minus_iCAF,112,18.7,0.071,5.11,-1.02,weak/outlier-driven


In [92]:
labels = js["consensus"][3]["labels"]
labels

T-C3L-00277       1
T-C3L-00589       1
T-C3L-00625       1
T-C3L-00640       1
T-C3L-00819       1
                 ..
T-TCGA-US-A774    1
T-TCGA-US-A77E    1
T-TCGA-US-A77J    2
T-TCGA-XN-A8T5    2
T-TCGA-YH-A8SY    1
Name: k3, Length: 112, dtype: int32

In [93]:
s2 = labels[labels==2].index
dffib = mc.df_theta.loc[s2, cmap["fibroblast"]]     # low fibroblast content?
dffib

T-C3N-01719       0.858
T-TCGA-HZ-7920    0.245
T-TCGA-HZ-A49H    0.219
T-TCGA-IB-7897    0.463
T-TCGA-IB-AAUW    0.170
T-TCGA-Q3-A5QY    0.324
T-TCGA-US-A77J    0.392
T-TCGA-XN-A8T5    0.567
Name: Fibroblast cell, dtype: float64

In [94]:
dffib.describe()

count    8.000
mean     0.405
std      0.226
min      0.170
25%      0.238
50%      0.358
75%      0.489
max      0.858
Name: Fibroblast cell, dtype: float64

In [95]:
small = labels[labels == 1].index
small

Index(['T-C3L-00277', 'T-C3L-00589', 'T-C3L-00625', 'T-C3L-00640', 'T-C3L-00819', 'T-C3L-00881',
       'T-C3L-01051', 'T-C3L-01124', 'T-C3L-01598', 'T-C3L-01689', 'T-C3L-01971', 'T-C3L-02701',
       'T-C3L-02890', 'T-C3L-03123', 'T-C3L-03632', 'T-C3L-03635', 'T-C3L-04080', 'T-C3L-04495',
       'T-C3L-04853', 'T-C3N-00511', 'T-C3N-01382', 'T-C3N-01383', 'T-C3N-01388', 'T-C3N-01899',
       'T-C3N-02589', 'T-C3N-02765', 'T-C3N-02944', 'T-C3N-02996', 'T-C3N-03006', 'T-C3N-03061',
       'T-C3N-03069', 'T-C3N-03173', 'T-C3N-03439', 'T-C3N-03665', 'T-C3N-03666', 'T-C3N-03754',
       'T-C3N-03839', 'T-TCGA-2J-AAB1', 'T-TCGA-2J-AAB4', 'T-TCGA-2J-AAB8', 'T-TCGA-2J-AABH',
       'T-TCGA-2J-AABT', 'T-TCGA-2L-AAQL', 'T-TCGA-3A-A9I9', 'T-TCGA-3A-A9IB', 'T-TCGA-3A-A9IH',
       'T-TCGA-3A-A9IL', 'T-TCGA-3A-A9IN', 'T-TCGA-3A-A9IV', 'T-TCGA-F2-6879', 'T-TCGA-F2-7276',
       'T-TCGA-F2-A44G', 'T-TCGA-FB-A78T', 'T-TCGA-FB-A7DR', 'T-TCGA-H6-8124', 'T-TCGA-H6-A45N',
       'T-TCGA-H8-A6C1', 'T-TCGA-

In [96]:
set(s2) & set(small)

set()

In [97]:
s2 = labels[labels == 2].index
mc.df_theta.loc[s2, cmap["fibroblast"]].sort_values().round(4)

T-TCGA-IB-AAUW    0.170
T-TCGA-HZ-A49H    0.219
T-TCGA-HZ-7920    0.245
T-TCGA-Q3-A5QY    0.324
T-TCGA-US-A77J    0.392
T-TCGA-IB-7897    0.463
T-TCGA-XN-A8T5    0.567
T-C3N-01719       0.858
Name: Fibroblast cell, dtype: float64

In [98]:
scores, cov = mc.program_scores(compartment_map=cmap, samples=X.index)
cov[["compartment","program","n_found","r_with_theta"]]

,compartment,program,n_found,r_with_theta
0,malignant,basal,12,0.0
1,malignant,classical,12,0.0
2,malignant,emt,6,0.0
3,malignant,prolif,6,0.0
4,fibroblast,myCAF,8,0.0
5,fibroblast,iCAF,9,0.0
6,fibroblast,apCAF,4,0.0
7,macrophage,M1,5,-0.0
8,macrophage,TAM,7,0.0
9,macrophage,SPP1,5,0.0


In [99]:
mc.axis_modality(scores)
js = mc.joint_states(scores)
mc.state_profile(js)

,malignant.axis_basal_minus_classical,fibroblast.axis_myCAF_minus_iCAF,n,mean_abs_score
k2,,,,
1,-0.292,0.219,92,0.569
2,1.342,-1.006,20,1.460


### Both cohorts and modality

The better use of two cohorts is replication, not deletion. Run the pipeline in each independently and keep what reproduces

An axis that is bimodal in both, or a basal↔myCAF coupling with the same sign in both, is far stronger evidence than anything from a merged analysis — and it doesn't require you to decide which cohort to trust.

If you want a discovery/validation split, use TCGA as discovery (n=68) and CPTAC as validation (n=44). Counterintuitive given TCGA is the noisier one, but discovery needs the power and validation needs the clean measurement.

If you do want to remove something, remove the specific samples rather than the cohort — the four with θ_fib < 0.05 from your sorted list, which min_theta=0.02 already partly handles. That's a stated QC criterion applied uniformly to both cohorts, which is defensible in a way that "we dropped TCGA" isn't.

One caveat on my own advice: TCGA-PAAD is known for low neoplastic cellularity in a substantial fraction of cases, so some of that extra dispersion is probably real data quality rather than batch. That argues for the QC threshold, not for dropping the cohort.


In [102]:
for name in ["TCGA", "CPTAC"]:
    idx = scores.index[coh == name]
    sc_c, cov_c = mc.program_scores(compartment_map=cmap, samples=idx)
    print(name, len(idx))
    print("------"*5)
    print(mc.axis_modality(sc_c))
    print("------"*5)
    print(mc.couple_compartments(sc_c, n_perm=1000).head(5))
    print("------"*5)


TCGA 73
------------------------------
                                   axis   n  delta_bic  min_component_weight  separation_sd  skew  \
0  malignant.axis_basal_minus_classical  73        7.4                 0.192           1.53  1.56   
1      fibroblast.axis_myCAF_minus_iCAF  68        9.6                 0.159           3.54 -1.01   

               verdict  
0  weak/outlier-driven  
1  weak/outlier-driven  
------------------------------
          program_a                         program_b      r     p_perm   n    fdr
0  malignant.prolif  fibroblast.axis_myCAF_minus_iCAF  0.446  9.990e-04  68  0.041
1  malignant.prolif                   fibroblast.iCAF -0.486  9.990e-04  68  0.041
2   malignant.basal             endothelial.lymphatic -0.433  8.991e-03  39  0.176
3     malignant.emt                    macrophage.TAM -0.469  9.990e-03  30  0.176
4   fibroblast.iCAF                   macrophage.SPP1 -0.408  1.399e-02  30  0.176
------------------------------
CPTAC 44
-------------

This is the payoff. Two clean results, one negative and one positive.

**Neither axis is bimodal in either cohort.** TCGA: ΔBIC 7.4 and 9.6, both below the threshold, both flagged outlier-driven. CPTAC: ΔBIC −1.8 and −9.6, both unimodal. That's a consistent, replicated negative — **discrete subtypes are not supported in your data.** Your basal/classical and myCAF/iCAF axes are continua. The k=3 joint states, the 2×2 cross-tab, the 12-sample "iCAF group" — all were partitions of clouds. Stop pursuing a taxonomy and report continuous axes.

**One coupling replicates across cohorts:**

| | TCGA | CPTAC |
|---|---|---|
| `malignant.prolif` ↔ `fibroblast.iCAF` | r = −0.486, FDR 0.041 | r = −0.497, FDR 0.050 |

Same sign, near-identical magnitude, independently significant in both, on n=68 and n=44. TCGA's top hit (`prolif` ↔ `myCAF_minus_iCAF`, r = +0.446) is the same relationship read on the contrast. And this is already θ-controlled — `couple_compartments` defaults to `control_theta=True`.

That's your tumour–stroma crosstalk finding: **proliferative malignant programs co-occur with iCAF-depleted stroma.** It's exactly the non-cell-autonomous signal you asked about eight turns ago, and it survived the one test that matters — independent replication.

Nothing else replicates. TCGA's `emt`↔`macrophage.TAM` and CPTAC's `emt`↔`myCAF`, `M1`↔`endothelial.activated`, `SPP1`↔`tip_angio` each appear in one cohort only.

Two cautions on those non-replicating hits. Note the `n` column: 27–39 for macrophage and endothelial pairs, versus 44–68 for fibroblast. Those are the `min_theta` drops, and a correlation of 0.65 at n=27 has a 95% CI roughly 0.35–0.82 — wide enough that failure to replicate is uninformative either way. And `compartment_readiness` on those low-θ compartments is worth checking before you interpret them at all.

For the writeup, the honest framing is: two continuous compartment axes, cohort-stable for tumour and heteroscedastic for stroma, with a single replicated inverse coupling between malignant proliferation and inflammatory CAF content.

### Delta-BIC

Delta BIC (ΔBIC) measures the difference between a candidate model's Bayesian information criterion BIC_{m} and the minimum BIC score {BIC}^{*} among a set of models. 

Calculated as Delta_BIC = {BIC}_m - {BIC}^*

it evaluates the relative evidence against a higher-scoring model. Lower BIC values indicate preferred, more parsimonious models.Interpreting Delta BIC Values0 to 2: Weak or bare difference; little to no evidence against the higher BIC model.2 to 6: Positive or moderate evidence favoring the model with the lower BIC.6 to 10: Strong evidence that the model with the lower BIC is superior.> 10: Very strong evidence against the higher BIC model, strongly supporting the minimum BIC choice.Watch this video for a high-level conceptual breakdown of how information criteria like AIC and BIC compare models and penalize complexity:17:25Multiple Regression, AIC, AICc, and BIC Basics59K views · 4 years agoYouTube · Brandon FoltzContext and UsageModel Selection: Used alongside maximum likelihood estimation to balance goodness of fit with model complexity while heavily penalizing extra parameters.Sample Size Sensitivity: Unlike AIC, BIC incorporates sample size (N) into its penalty term (\(K \ln(N)\)), making it more conservative as sample sizes increase.If you want to apply this, tell me:Are you comparing nested or non-nested models?What is your sample size and number of parameters?I can help you compute or interpret your specific values.

In [101]:
print(pml.__version__)

0.31.1
